# SI Fig 3 (Clark): Hyperparameter comparison & method comparison

Combines:
- Hyperparameter sweep: Sankey, R² strip plot, confusion matrices
- Method comparison: Metropolis, co-occurrence, PCA, FCNN baselines vs SCiFI

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tqdm.autonotebook import tqdm
from glob import glob
from sklearn.linear_model import LinearRegression
import scipy.cluster.hierarchy as shc
import plotly.graph_objects as go
import IPython

from utils.models import FC_Gumbelpredictor
from utils.metropolis import metropolis, compute_regression_loss
from dataset import ClarkDataset

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.Set1.colors)
plt.rcParams['figure.figsize'] = [3, 2]
plt.rcParams['figure.dpi'] = 200
plt.rcParams['svg.fonttype'] = 'none'

device = torch.device('cpu')

## Load all models

In [ ]:
masterroot = './trained_models'

subroots = [
    'gumbel_241218_140330_N-256_L-2_DO-0._LR-1e-1_TAU-1._0.995_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Acetate_DI-0._1',
    'gumbel_241218_140331_N-256_L-2_DO-0._LR-1e-1_TAU-1._0.995_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Butyrate_DI-0._0',
    'gumbel_241218_153144_N-256_L-2_DO-0._LR-1e-1_TAU-1._0.995_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Succinate_DI-0._2',
    'gumbel_241218_153144_N-256_L-2_DO-0._LR-1e-1_TAU-1._0.999_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Acetate_DI-0._4',
    'gumbel_241218_153144_N-256_L-2_DO-0._LR-1e-1_TAU-1._0.999_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Butyrate_DI-0._3',
    'gumbel_241218_153144_N-256_L-2_DO-0._LR-1e-1_TAU-1._0.999_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Succinate_DI-0._5',
    'gumbel_241218_153633_N-256_L-2_DO-0._LR-1e-2_TAU-1._0.995_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Butyrate_DI-0._6',
    'gumbel_241218_153744_N-256_L-2_DO-0._LR-1e-2_TAU-1._0.995_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Acetate_DI-0._7',
    'gumbel_241218_153744_N-256_L-2_DO-0._LR-1e-2_TAU-1._0.995_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Succinate_DI-0._8',
    'gumbel_241218_160450_N-256_L-2_DO-0._LR-1e-2_TAU-1._0.999_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Butyrate_DI-0._9',
    'gumbel_241218_163312_N-256_L-2_DO-0._LR-1e-2_TAU-1._0.999_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Acetate_DI-0._10',
    'gumbel_241218_163351_N-256_L-2_DO-0._LR-1e-2_TAU-1._0.999_0.1_SCH-0.999_OPT-adam_ACT-gelu_NENS-20_Nsteps-10000_OUT-Succinate_DI-0._11'
]

roots = [os.path.join(masterroot, sr) for sr in subroots]

In [ ]:
def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true, axis=0)) ** 2)

loss_df = []
dataset_train = None

for sr in tqdm(subroots):
    root = os.path.join(masterroot, sr)
    for file in os.listdir(root):
        npcs = int(file.split('_')[0].split('-')[-1])
        ensidx = int(file.split('_')[1].split('-')[-1].split('.')[0])

        path = os.path.join(root, file)
        x = torch.load(path, map_location=device)

        if dataset_train is None:
            train_kwargs = dict(x['train_dataset_kwargs'])
            train_kwargs['device'] = device
            dataset_train = ClarkDataset(**train_kwargs)

        pred = np.asarray(x.get('pred_train', []))
        targ = np.asarray(x.get('target_train', []))
        test_pred = np.asarray(x.get('pred_test', []))
        test_targ = np.asarray(x.get('target_test', []))

        tauparams = {'tau_' + k: v for k, v in x.get('tau_hparams', {}).items()}

        loss_df.append({
            'npcs': npcs, 'ensidx': ensidx,
            'tau': x.get('curr_tau', np.nan),
            'train_loss': x.get('train_losses', []), 'test_loss': x.get('test_losses', []),
            'train_preds': pred.squeeze() if pred.size else pred,
            'train_targets': targ.squeeze() if targ.size else targ,
            'test_preds': test_pred.squeeze() if test_pred.size else test_pred,
            'test_targets': test_targ.squeeze() if test_targ.size else test_targ,
            'r2_test': x.get('r2_test', np.nan),
            'r2_train': x.get('r2_train', np.nan),
            'proj': np.asarray(x.get('proj', np.nan)),
            'proj_det': np.asarray(x.get('proj_det', np.nan)),
            'logits': np.asarray(x.get('proj_logits', np.nan)),
            'lr': x.get('model_kwargs', {}).get(1, {}).get('optimizer_hparams', {}).get('LR', np.nan),
            'dropout': x.get('model_kwargs', {}).get(1, {}).get('network_hparams', {}).get('dropout', np.nan),
            'dropout_input': x.get('model_kwargs', {}).get(1, {}).get('network_hparams', {}).get('dropout_input', np.nan),
            'output': x.get('test_dataset_kwargs', {}).get('output', ''),
            'loc': path,
            **tauparams
        })

loss_df = pd.DataFrame(loss_df)
names = np.array(dataset_train.names)
abd = dataset_train.inputs.detach().cpu().numpy()

print('loss_df shape:', loss_df.shape)
print('Outputs:', loss_df.output.unique())

## Part 1: Hyperparameter comparison
### Enumerate hyperparameter settings

In [ ]:
def sorted_unique(col):
    vals = pd.Series(loss_df[col]).dropna().unique().tolist()
    try:
        return np.array(sorted(vals))
    except Exception:
        return np.array(sorted(vals, key=lambda z: str(z)), dtype=object)

hparam_cols = ['lr', 'dropout', 'dropout_input', 'tau_init', 'tau_relax_rate', 'tau_min', 'npcs', 'output']
available_hparams = {k: sorted_unique(k) for k in hparam_cols if k in loss_df.columns}

npc_fixed = 4
output_fixed = 'Butyrate'
n_ensemble_keep = 10
THRESHOLD = 0.7

vary_keys = [k for k in ['lr', 'dropout_input', 'tau_relax_rate', 'tau_init']
             if k in available_hparams and len(available_hparams[k]) > 0]
baseline = {k: available_hparams[k][0] for k in vary_keys}

setting_list = [baseline.copy()]
for k in vary_keys:
    for val in available_hparams[k]:
        h = baseline.copy()
        h[k] = val
        setting_list.append(h)

seen = set()
setting_list_unique = []
for h in setting_list:
    key = tuple((k, h[k]) for k in vary_keys)
    if key not in seen:
        seen.add(key)
        setting_list_unique.append(h)
setting_list = setting_list_unique

def label_from_setting(h):
    return ', '.join([f'{k}={h[k]}' for k in vary_keys])

print(f'{len(setting_list)} unique settings')
for i, h in enumerate(setting_list):
    print(i, h)

### Consensus grouping across hyperparameter settings

In [ ]:
assignment_full = []
keep_masks = []
setting_labels_kept = []
r2_values_by_setting = []
mean_r2_by_setting = []

# Find common species across all settings
common_keep = np.ones(len(names), dtype=bool)

for hparams in setting_list:
    mask = (loss_df.npcs == npc_fixed) & (loss_df.output == output_fixed)
    for k in vary_keys:
        mask = mask & (loss_df[k] == hparams[k])

    r2_vals = loss_df.loc[mask, 'r2_test'].values
    if len(r2_vals) == 0:
        continue

    ens_sorted = loss_df.loc[mask, 'ensidx'].values[np.argsort(r2_vals)[::-1]]
    ens_sorted = ens_sorted[:n_ensemble_keep]

    all_projs = np.zeros((len(ens_sorted), len(names), npc_fixed))
    for e, ens_idx in enumerate(ens_sorted):
        proj = loss_df.loc[mask & (loss_df.ensidx == ens_idx), 'proj_det'].values[0].copy()
        proj[np.arange(len(proj)), np.argmax(proj, axis=1)] = 1
        proj[proj < 1] = 0

        corrs = np.zeros(npc_fixed)
        for i in range(npc_fixed):
            intragp_abd = abd * proj[None, :, i]
            corrs[i] = np.mean(np.mean(intragp_abd, axis=1))
        group_sort = np.argsort(corrs)[::-1]
        all_projs[e] = all_projs[e][:, group_sort] if npc_fixed > 1 else all_projs[e]

    mean_assignments = np.mean(all_projs, axis=0)
    ids_full = np.argmax(mean_assignments, axis=1).astype(int)

    assignment_full.append(ids_full)
    keep_masks.append(common_keep.copy())
    setting_labels_kept.append(label_from_setting(hparams))
    r2_values_by_setting.append(np.asarray(r2_vals).astype(float))

    n_top = min(n_ensemble_keep, len(r2_vals))
    r2_top = np.sort(np.asarray(r2_vals).astype(float))[::-1][:n_top]
    mean_r2_by_setting.append(np.nanmean(r2_top))

# Sort by mean R2
sort_idx = np.argsort(mean_r2_by_setting)[::-1]
assignment_full = [assignment_full[i] for i in sort_idx]
setting_labels_kept = [setting_labels_kept[i] for i in sort_idx]
r2_values_by_setting = [r2_values_by_setting[i] for i in sort_idx]
mean_r2_by_setting = [mean_r2_by_setting[i] for i in sort_idx]

assignment_lists = [ids.copy().astype(int) for ids in assignment_full]

# Color species by reference setting
palette = ['blue', 'green', 'red', 'orange', 'purple', 'brown', 'pink', 'olive']
ids_ref = assignment_lists[0]
species_colors = np.full(len(names), 'lightgray', dtype=object)
for g in range(npc_fixed):
    species_colors[ids_ref == g] = palette[g % len(palette)]

assignment_lists_shifted = []
for ids in assignment_lists:
    ids_shifted = ids.copy()
    if len(assignment_lists_shifted) > 0:
        ids_shifted = ids_shifted + assignment_lists_shifted[-1].max() + 1
    assignment_lists_shifted.append(ids_shifted)

print(f'{len(assignment_lists)} valid settings')

### SI: Sankey diagram across hyperparameter settings

In [ ]:
species_mean_abd = abd.mean(axis=0)

srcs, trgs, link_names, link_colors_all, widths = [], [], [], [], []
for i in range(len(assignment_lists_shifted) - 1):
    srcs += assignment_lists_shifted[i].tolist()
    trgs += assignment_lists_shifted[i + 1].tolist()
    link_names += names.tolist()
    link_colors_all += species_colors.tolist()
    widths += species_mean_abd.tolist()

node_values = np.unique(np.concatenate(assignment_lists_shifted))
node_map = {v: i for i, v in enumerate(node_values)}
source = [node_map[s] for s in srcs]
target = [node_map[t] for t in trgs]

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), color='lightblue'),
    link=dict(source=source, target=target, value=widths, label=link_names,
              line=dict(width=0), color=link_colors_all)
)])
fig.update_layout(height=650, width=700)
fig.show()

if input('save? ').strip().lower() == 'save':
    fig.write_image('./figures/SI_clark_compare_hparams_sankey.svg')

### SI: R² distribution across hyperparameter settings

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(2.2 * len(setting_labels_kept), 2), dpi=200)
jitter = 0.10
for i, r2_vals in enumerate(r2_values_by_setting):
    r2_vals = np.asarray(r2_vals).astype(float)
    order = np.argsort(r2_vals)[::-1]
    top_idx = order[:min(n_ensemble_keep, len(order))]
    rest_idx = order[min(n_ensemble_keep, len(order)):]
    if len(rest_idx) > 0:
        x_rest = i + np.random.uniform(-jitter, jitter, size=len(rest_idx))
        ax.scatter(x_rest, r2_vals[rest_idx], s=14, alpha=0.45, color='gray', lw=0,
                   label='other' if i == 0 else None)
    x_top = i + np.random.uniform(-jitter, jitter, size=len(top_idx))
    ax.scatter(x_top, r2_vals[top_idx], s=20, alpha=0.9, color='crimson', lw=0,
               label=f'top {n_ensemble_keep}' if i == 0 else None)

ax.set_ylim(0, 1)
ax.set_xticks(np.arange(len(setting_labels_kept)))
ax.set_xticklabels(setting_labels_kept, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('R² test')
ax.legend(frameon=False, fontsize=7)
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()

if input('save? ').strip().lower() == 'save':
    fig.savefig('./figures/SI_clark_compare_hparams_R2.svg', bbox_inches='tight')

### SI: Pairwise confusion matrices

In [ ]:
n_sets = len(assignment_lists)
conf_mats = {}
vmax = 0
for a in range(n_sets):
    for b in range(n_sets):
        ua = np.unique(assignment_lists[a])
        ub = np.unique(assignment_lists[b])
        mat = np.zeros((len(ua), len(ub)), dtype=int)
        for i, ga in enumerate(ua):
            for j, gb in enumerate(ub):
                mat[i, j] = np.sum((assignment_lists[a] == ga) & (assignment_lists[b] == gb))
        conf_mats[(a, b)] = mat
        vmax = max(vmax, mat.max())

fig, ax = plt.subplots(n_sets, n_sets, figsize=(0.8 * n_sets, 0.8 * n_sets), dpi=200)
if n_sets == 1:
    ax = np.array([[ax]])

for a in range(n_sets):
    for b in range(n_sets):
        this_ax = ax[a, b]
        if a > b:
            this_ax.axis('off')
            continue
        mat = conf_mats[(a, b)]
        this_ax.imshow(mat, cmap='binary', vmin=0, vmax=vmax)
        for i in range(mat.shape[1]):
            for j in range(mat.shape[0]):
                color = 'white' if mat[j, i] > vmax / 2 else 'k'
                this_ax.text(i, j, f'{mat[j, i]}', color=color, ha='center', va='center', fontsize=6)
        this_ax.set_xticks([])
        this_ax.set_yticks([])

fig.suptitle('Pairwise confusion matrices', fontsize=9)
fig.tight_layout(pad=0.2)

if input('save? ').strip().lower() == 'save':
    fig.savefig('./figures/SI_clark_compare_hparams_confusion.svg', bbox_inches='tight')

## Part 2: Method comparison (Metropolis, co-occurrence, PCA, FCNN)
### Select best SCiFI models

In [ ]:
compare_hparams = {'dropout_input': 0.0, 'dropout': 0.0, 'lr': 0.01,
                   'output': 'Butyrate', 'tau_relax_rate': 0.999, 'npc': 4}

mask = ((loss_df.npcs == compare_hparams['npc']) &
        (loss_df.lr == compare_hparams['lr']) &
        (loss_df.dropout_input == compare_hparams['dropout_input']) &
        (loss_df.output == compare_hparams['output']) &
        (loss_df.tau_relax_rate == compare_hparams['tau_relax_rate']))

loss_df_red = loss_df[mask].sort_values('r2_test', ascending=False)
paths = loss_df_red['loc'].values

print(f'{len(paths)} models sorted by R2 test')
print('Top 5 R2:', loss_df_red['r2_test'].values[:5])

### Run Metropolis baseline

In [ ]:
import time

def p_mat_from_idx_list(grouping):
    n_groups = len(set(grouping))
    n_species = len(grouping)
    P = np.zeros((n_species, n_groups))
    for i, g in enumerate(grouping):
        P[i, g] = 1
    return P

metropolis_results = []
n_ens = min(10, len(paths))

for ensidx, path in enumerate(tqdm(paths[:n_ens])):
    x = torch.load(path, map_location=device)
    ds_train = ClarkDataset(**x['train_dataset_kwargs'])
    ds_test = ClarkDataset(**x['test_dataset_kwargs'])
    sample_train = next(iter(ds_train.loader_list[ensidx]))
    sample_test = next(iter(ds_test.loader_list[ensidx]))

    abd_train = sample_train['input'].detach().cpu().numpy()
    abd_test = sample_test['input'].detach().cpu().numpy()
    targ_train = sample_train['target'].detach().cpu().numpy()
    targ_test = sample_test['target'].detach().cpu().numpy()

    def loss_fct_train(grouping):
        P = p_mat_from_idx_list(grouping)
        return compute_regression_loss(abd_train @ P, targ_train)

    t0 = time.time()
    best_grouping, best_loss, loss_trace = metropolis(
        loss_fct_train, n_species=len(names), n_groups=compare_hparams['npc'],
        n_steps=10000, beta_init=1.0, verbose=False)
    elapsed = time.time() - t0

    P_best = p_mat_from_idx_list(best_grouping)
    reg = LinearRegression().fit(abd_train @ P_best, targ_train)
    r2_met_test = reg.score(abd_test @ P_best, targ_test)

    metropolis_results.append({
        'ensidx': ensidx, 'r2_test': r2_met_test,
        'grouping': best_grouping, 'time': elapsed
    })

metropolis_results = pd.DataFrame(metropolis_results)
print('Metropolis mean R2:', metropolis_results.r2_test.mean())

### Co-occurrence hierarchical clustering

In [ ]:
abd_norm = abd / (abd.sum(axis=1, keepdims=True) + 1e-12)
corr_abd = np.corrcoef(abd_norm.T)
np.fill_diagonal(corr_abd, 1.0)
dists = 1.0 - corr_abd
dists = np.clip(dists, 0, None)
upper = dists[np.triu_indices(dists.shape[0], k=1)]

Z = shc.linkage(upper, method='ward')
grouping_coocc = shc.fcluster(Z, t=compare_hparams['npc'], criterion='maxclust') - 1
P_coocc = p_mat_from_idx_list(grouping_coocc)

print('Co-occurrence groups:', [np.sum(grouping_coocc == g) for g in range(compare_hparams['npc'])])

### PCA projection

In [ ]:
u, s, vh = np.linalg.svd(abd, full_matrices=False)
pca_proj_mat = vh[:compare_hparams['npc'], :].T
print('PCA projection shape:', pca_proj_mat.shape)

### Train FCNN baselines (co-occ + PCA groupings)

In [ ]:
class FCNN(nn.Module):
    def __init__(self, grouping_mat, channels, lr=1e-3):
        super().__init__()
        self.register_buffer('grouping_mat', torch.from_numpy(grouping_mat).float())
        layers = []
        for i in range(len(channels) - 1):
            layers.append(nn.Linear(channels[i], channels[i + 1]))
            if i < len(channels) - 2:
                layers.append(nn.GELU())
        self.net = nn.Sequential(*layers)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    def forward(self, x):
        x = x @ self.grouping_mat
        return self.net(x)


channels = [compare_hparams['npc'], 256, 256, 1]
ens_size = min(10, len(paths))
n_epochs = 2000

coocc_r2, pc_r2 = [], []

for ensidx in tqdm(range(ens_size)):
    x = torch.load(paths[ensidx], map_location=device)
    ds_train = ClarkDataset(**x['train_dataset_kwargs'])
    ds_test = ClarkDataset(**x['test_dataset_kwargs'])
    sample_train = next(iter(ds_train.loader_list[ensidx]))
    sample_test = next(iter(ds_test.loader_list[ensidx]))

    for proj_name, proj_mat, r2_list in [('coocc', P_coocc, coocc_r2), ('pca', pca_proj_mat, pc_r2)]:
        model = FCNN(proj_mat, channels, lr=1e-3).to(device)
        X_tr = sample_train['input'].to(device)
        Y_tr = sample_train['target'].to(device).unsqueeze(-1)
        X_te = sample_test['input'].to(device)
        Y_te = sample_test['target'].to(device).unsqueeze(-1)

        for epoch in range(n_epochs):
            model.train()
            model.optimizer.zero_grad()
            loss = nn.MSELoss()(model(X_tr), Y_tr)
            loss.backward()
            model.optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(X_te).cpu().numpy().squeeze()
            targ = Y_te.cpu().numpy().squeeze()
        r2_list.append(safe_r2(targ, pred))

print('Co-occ NN mean R2:', np.nanmean(coocc_r2))
print('PC NN mean R2:', np.nanmean(pc_r2))

### SI: Method comparison bar chart

In [ ]:
# Gather all R2 values
scifi_r2 = loss_df_red['r2_test'].values[:ens_size]

# Co-occurrence LR
coocc_lr_r2 = []
for ensidx in range(ens_size):
    x = torch.load(paths[ensidx], map_location=device)
    ds_train = ClarkDataset(**x['train_dataset_kwargs'])
    ds_test = ClarkDataset(**x['test_dataset_kwargs'])
    sample_train = next(iter(ds_train.loader_list[ensidx]))
    sample_test = next(iter(ds_test.loader_list[ensidx]))

    abd_tr = sample_train['input'].detach().cpu().numpy()
    abd_te = sample_test['input'].detach().cpu().numpy()
    y_tr = sample_train['target'].detach().cpu().numpy()
    y_te = sample_test['target'].detach().cpu().numpy()

    reg = LinearRegression().fit(abd_tr @ P_coocc, y_tr)
    coocc_lr_r2.append(reg.score(abd_te @ P_coocc, y_te))

# SCiFI LR
scifi_lr_r2 = []
for ensidx in range(ens_size):
    ens_mask = mask & (loss_df.ensidx == loss_df_red.ensidx.values[ensidx])
    proj = loss_df.loc[ens_mask, 'proj_det'].values[0].copy()
    proj[np.arange(len(proj)), np.argmax(proj, axis=1)] = 1
    proj[proj < 1] = 0

    x = torch.load(paths[ensidx], map_location=device)
    ds_train = ClarkDataset(**x['train_dataset_kwargs'])
    ds_test = ClarkDataset(**x['test_dataset_kwargs'])
    sample_train = next(iter(ds_train.loader_list[ensidx]))
    sample_test = next(iter(ds_test.loader_list[ensidx]))

    abd_tr = sample_train['input'].detach().cpu().numpy()
    abd_te = sample_test['input'].detach().cpu().numpy()
    y_tr = sample_train['target'].detach().cpu().numpy()
    y_te = sample_test['target'].detach().cpu().numpy()

    reg = LinearRegression().fit(abd_tr @ proj, y_tr)
    scifi_lr_r2.append(reg.score(abd_te @ proj, y_te))

methods = ['Co-occ LR', 'SCiFI LR', 'Metr.', 'Co-occ+NN', 'PC+NN', 'SCiFI']
all_r2 = [coocc_lr_r2, scifi_lr_r2, metropolis_results.r2_test.values,
           coocc_r2, pc_r2, scifi_r2]

fig, ax = plt.subplots(1, 1, figsize=(4, 2.5))
for i, (label, vals) in enumerate(zip(methods, all_r2)):
    vals = np.asarray(vals).astype(float)
    jit = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax.scatter(i + jit, vals, s=15, alpha=0.6, lw=0)
    ax.scatter(i, np.nanmean(vals), s=40, marker='_', color='k', lw=2)

ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=7)
ax.set_ylabel('R² test')
ax.set_ylim(-0.2, 1.0)
ax.tick_params(direction='in')
fig.tight_layout()

if input('save? ').strip().lower() == 'save':
    fig.savefig('./figures/SI_clark_method_comparison.svg', bbox_inches='tight')

### SI: SCiFI vs Metropolis vs Co-occurrence confusion matrices

In [ ]:
# SCiFI consensus
scifi_all_projs = all_projs_by_npc = {}
npc = compare_hparams['npc']
mask_s = ((loss_df.npcs == npc) & (loss_df.lr == compare_hparams['lr']) &
          (loss_df.dropout_input == compare_hparams['dropout_input']) &
          (loss_df.output == compare_hparams['output']) &
          (loss_df.tau_relax_rate == compare_hparams['tau_relax_rate']))

test_r2_s = loss_df.loc[mask_s, 'r2_test'].values
ens_sorted_s = loss_df.loc[mask_s, 'ensidx'].values[np.argsort(test_r2_s)[::-1]][:10]

all_projs_s = np.zeros((len(ens_sorted_s), len(names), npc))
for e, ens_idx in enumerate(ens_sorted_s):
    proj = loss_df.loc[mask_s & (loss_df.ensidx == ens_idx), 'proj_det'].values[0].copy()
    proj[np.arange(len(proj)), np.argmax(proj, axis=1)] = 1
    proj[proj < 1] = 0
    corrs = np.array([np.mean(abd * proj[None, :, i]) for i in range(npc)])
    all_projs_s[e] = proj[:, np.argsort(corrs)[::-1]]

scifi_consensus = np.argmax(np.mean(all_projs_s, axis=0), axis=1)

# Metropolis consensus
mc_groupings = [r['grouping'] for _, r in metropolis_results.iterrows()]
mc_all_projs = np.array([p_mat_from_idx_list(g) for g in mc_groupings])
for e in range(len(mc_all_projs)):
    corrs = np.array([np.mean(abd * mc_all_projs[e, :, i:i+1]) for i in range(npc)])
    mc_all_projs[e] = mc_all_projs[e][:, np.argsort(corrs)[::-1]]
mc_consensus = np.argmax(np.mean(mc_all_projs, axis=0), axis=1)

fig, ax = plt.subplots(1, 2, figsize=(5, 2), dpi=200)

for idx, (name, other) in enumerate([('Metropolis', mc_consensus), ('Co-occurrence', grouping_coocc)]):
    mat = np.zeros((npc, npc), dtype=int)
    for i in range(npc):
        for j in range(npc):
            mat[i, j] = np.sum((scifi_consensus == i) & (other == j))
    ax[idx].imshow(mat, cmap='binary')
    for i in range(npc):
        for j in range(npc):
            color = 'white' if mat[i, j] > mat.max() / 2 else 'k'
            ax[idx].text(j, i, str(mat[i, j]), ha='center', va='center', color=color, fontsize=8)
    ax[idx].set_title(f'SCiFI vs {name}', fontsize=8)
    ax[idx].set_xlabel(name)
    ax[idx].set_ylabel('SCiFI')

fig.tight_layout()

if input('save? ').strip().lower() == 'save':
    fig.savefig('./figures/SI_clark_scifi_vs_baselines_confusion.svg', bbox_inches='tight')